In [ ]:
import sys
sys.path.append('..')

import json
import joblib
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

from src.parser import parse_ssh_fields, build_miner, mine_templates
from src.features import build_windows, count_vector
from src.explain import build_baseline

In [ ]:
df = parse_ssh_fields('../data/raw/SSH_full/SSH.log')
print('parsed rows:', len(df))

miner = build_miner('../drain3_ssh.ini', '../models/ssh_drain_state_full.bin')
df = mine_templates(df, miner)
df.to_csv('../data/parsed/ssh_parsed_full.csv', index=False)
print('unique templates:', df['EventId'].nunique())

In [ ]:
windows = build_windows(df, window_size=100)
print('windows:', len(windows))

X_counts, event_names = count_vector(windows)
print('X_counts shape:', X_counts.shape)
np.save('../data/features/X_counts_full_ssh.npy', X_counts)

templates = df[['EventId','EventTemplate']].drop_duplicates(subset='EventId').sort_values('EventId')
templates.to_csv('../data/parsed/templates_full_ssh.csv', index=False)

In [ ]:
X_counts = np.load('../data/features/X_counts_full_ssh.npy')
CONTAMINATION = 0.05

pca_boot = PCA(n_components=5, random_state=42).fit(X_counts)
recon = pca_boot.inverse_transform(pca_boot.transform(X_counts))
boot_errors = np.mean((X_counts - recon) ** 2, axis=1)
boot_threshold = np.percentile(boot_errors, 100 * (1 - CONTAMINATION))
y_bootstrap = (boot_errors > boot_threshold).astype(int)
X_normal = X_counts[y_bootstrap == 0]
print('bootstrap flagged as anomalous:', y_bootstrap.sum(), '/', len(y_bootstrap))
print('X_normal shape:', X_normal.shape)

In [ ]:
pca = PCA(n_components=5, random_state=42)
pca.fit(X_normal)
reconstructed = pca.inverse_transform(pca.transform(X_normal))
errors = np.mean((X_normal - reconstructed) ** 2, axis=1)
pca_threshold = float(np.percentile(errors, 100 * (1 - CONTAMINATION)))
joblib.dump(pca, '../models/ssh/pca.joblib')
print('PCA threshold:', round(pca_threshold, 6))

iso = IsolationForest(contamination=CONTAMINATION, random_state=42)
iso.fit(X_normal)
joblib.dump(iso, '../models/ssh/isolation_forest.joblib')

rng = np.random.default_rng(42)
idx = rng.choice(len(X_normal), size=min(50_000, len(X_normal)), replace=False)
lof = LocalOutlierFactor(n_neighbors=20, contamination=CONTAMINATION, novelty=True)
lof.fit(X_normal[idx])
joblib.dump(lof, '../models/ssh/lof.joblib')
print('saved pca / isolation_forest / lof')

## Save metadata

In [ ]:
baseline = build_baseline(X_counts, y_bootstrap)
np.save('../models/ssh/baseline.npy', baseline)

templates = pd.read_csv('../data/parsed/templates_full_ssh.csv', keep_default_na=False)
event_names = templates['EventId'].tolist()

metadata = {
    'log_type': 'ssh',
    'event_names': [int(e) for e in event_names],
    'templates': {int(r.EventId): r.EventTemplate for r in templates.itertuples()},
    'pca_threshold': pca_threshold,
    'anomaly_rate': float(y_bootstrap.mean()),
    'trained_windows': int(len(X_normal)),
    'window_size': 100,
}
with open('../models/ssh/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print('saved metadata with', len(event_names), 'events')